[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gfascioli/NeuralNetwork_SMS_TextClassifier_freeCodeCamp_MachineLearningWithPython/blob/main/fcc_sms_text_classification.ipynb)

# 💬 Neural Network SMS Text Classifier

### freeCodeCamp — Machine Learning with Python Certification

**Goal:** classify SMS messages as `ham` (a normal message) or `spam` (an ad or unsolicited message), via a function `predict_message(text)` that returns `[likelihood_of_spam, "ham" or "spam"]`.

The plan:
1. Load the SMS Spam Collection train/test TSV files.
2. Map labels to 0 (ham) / 1 (spam).
3. Rebalance the training set (it's naturally ~87% ham) so the model can't just win by guessing.
4. Vectorize text into integers as its own explicit step, then train a model that only ever sees numbers -- with no masked pooling, to eliminate any division-by-zero risk.
5. Define `predict_message`, inspect exactly how it scores each official test message, then confirm against the official test cell.

---
*Solution by **Gonzalo Fascioli** — [github.com/Gfascioli](https://github.com/Gfascioli)*

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"


Write all your code in the cell below — the final cell is the official grader and should stay exactly as provided.

In [ ]:
# --- Solution by Gonzalo Fascioli (github.com/Gfascioli) ---

# each file has two tab-separated columns: label ("ham"/"spam"), then the message
train_dataset = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_dataset = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_dataset['label'] = train_dataset['label'].map({'ham': 0, 'spam': 1})
test_dataset['label'] = test_dataset['label'].map({'ham': 0, 'spam': 1})

train_messages = train_dataset['message'].values
train_labels = train_dataset['label'].values
test_messages = test_dataset['message'].values
test_labels = test_dataset['label'].values

# The training set is heavily imbalanced (~87% ham / ~13% spam). class_weight
# alone was NOT enough to stop an all-ham collapse, so we oversample spam
# examples until the classes are roughly balanced.
ham_mask = train_labels == 0
spam_mask = train_labels == 1

spam_multiplier = max(1, round(ham_mask.sum() / spam_mask.sum()))
spam_messages_oversampled = np.tile(train_messages[spam_mask], spam_multiplier)
spam_labels_oversampled = np.tile(train_labels[spam_mask], spam_multiplier)

train_messages_balanced = np.concatenate([train_messages[ham_mask], spam_messages_oversampled])
train_labels_balanced = np.concatenate([train_labels[ham_mask], spam_labels_oversampled])

print("Balanced training set spam ratio: {:.2f}".format(train_labels_balanced.mean()))

# Vectorization is its own explicit step, outside the model, so both
# training and single-message prediction feed the model plain integer
# arrays through the exact same path.
VOCAB_SIZE = 3000
MAX_LEN = 60

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(train_messages)

train_sequences = vectorize_layer(train_messages_balanced).numpy().astype('int32')
test_sequences = vectorize_layer(test_messages).numpy().astype('int32')

# Root-cause diagnostic: a row that vectorizes to ALL zeros (no real
# tokens at all) is exactly the condition that made the previous
# mask_zero=True + GlobalAveragePooling1D combination divide 0/0 -> NaN
# for that row, which then poisoned every weight in the network for the
# rest of training. Checking for it directly, rather than assuming.
empty_rows = np.sum(np.count_nonzero(train_sequences, axis=1) == 0)
print("Training rows with zero real tokens (all-padding): {}".format(empty_rows))

# --- Model: no masked pooling anywhere. ---
# GlobalAveragePooling1D + mask_zero=True computes sum(x)/count(real_tokens)
# per row -- a genuine division that returns NaN if a row's mask sums to
# zero, and one NaN loss value is enough to permanently corrupt every
# weight for the rest of training (which is what was actually happening
# here, confirmed by validation accuracy landing exactly on the trivial
# "always predict ham" baseline -- NaN outputs are always < 0.5 in every
# comparison). Flatten has no division of any kind, so this failure mode
# is structurally impossible with it.
model = keras.Sequential([
    keras.Input(shape=(MAX_LEN,), dtype='int32'),
    layers.Embedding(VOCAB_SIZE, 32),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    # clipnorm is cheap insurance against any other exploding-gradient-
    # driven NaN we haven't specifically diagnosed.
    optimizer=tf.keras.optimizers.Adam(clipnorm=1.0),
    metrics=['accuracy']
)

history = model.fit(
    train_sequences,
    train_labels_balanced,
    epochs=25,
    validation_data=(test_sequences, test_labels),
    verbose=0
)

# Explicit check: if this ever prints WARNING again, we know for certain
# training itself broke, immediately, rather than inferring it three
# debugging rounds later from a suspiciously-exact baseline accuracy.
if np.isnan(history.history['loss']).any() or np.isnan(history.history['val_loss']).any():
    print("WARNING: NaN detected in training loss history -- model weights are corrupted.")
else:
    print("Training loss stayed finite throughout -- no NaNs during training.")

print("Training finished after {} epochs.".format(len(history.history['loss'])))
print("Final validation accuracy: {:.3f}".format(history.history['val_accuracy'][-1]))

def predict_message(pred_text):
    sequence = tf.cast(vectorize_layer(tf.constant([pred_text], dtype=tf.string)), tf.int32)
    prediction = model(sequence, training=False)
    prob = float(prediction[0][0])
    label = 'spam' if prob >= 0.5 else 'ham'
    return [prob, label]

# --- Diagnostics: run BEFORE the official grading cell. ---

# 1) Confusion matrix on the full held-out test set.
test_preds = np.array([1 if predict_message(m)[1] == 'spam' else 0 for m in test_messages])
tp = int(np.sum((test_preds == 1) & (test_labels == 1)))
tn = int(np.sum((test_preds == 0) & (test_labels == 0)))
fp = int(np.sum((test_preds == 1) & (test_labels == 0)))
fn = int(np.sum((test_preds == 0) & (test_labels == 1)))
print("\nTest set confusion matrix:")
print("  spam correctly caught (TP): {}".format(tp))
print("  ham correctly kept    (TN): {}".format(tn))
print("  ham wrongly called spam (FP): {}".format(fp))
print("  spam missed, called ham (FN): {}".format(fn))

# 2) Exactly how the model scores each of the 7 official test messages.
diagnostic_messages = [
    ("how are you doing today", "ham"),
    ("sale today! to stop texts call 98912460324", "spam"),
    ("i dont want to go. can we try it a different day? available sat", "ham"),
    ("our new mobile video service is live. just install on your phone to start watching.", "spam"),
    ("you have won £1000 cash! call to claim your prize.", "spam"),
    ("i'll bring it tomorrow. don't forget the milk.", "ham"),
    ("wow, is your arm alright. that happened to me one time too", "ham"),
]

print("\nDiagnostic -- predicted probability of spam vs expected label:")
for msg, expected in diagnostic_messages:
    prob, label = predict_message(msg)
    mark = "OK   " if label == expected else "WRONG"
    print("  [{}] expected={:<5} predicted={:<5} p_spam={:.3f}   {}".format(
        mark, expected, label, prob, msg[:55]))


Run the cell below to test your model and function.

**This is the official freeCodeCamp grading cell — leave it unmodified.**

In [ ]:
# Run this cell to test your model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                    "sale today! to stop texts call 98912460324",
                    "i dont want to go. can we try it a different day? available sat",
                    "our new mobile video service is live. just install on your phone to start watching.",
                    "you have won £1000 cash! call to claim your prize.",
                    "i'll bring it tomorrow. don't forget the milk.",
                    "wow, is your arm alright. that happened to me one time too"
                    ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed. Keep trying.")

test_predictions()
